# Data Analysis — 20 Preguntas de Negocio

Todas las consultas sobre `ANALYTICS.OBT_TRIPS` usando Spark.

## Imports y configuración

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

spark = SparkSession.builder \
    .appName("P3_DataAnalysis") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

df = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "OBT_TRIPS") \
    .load()

df = df.toDF(*[c.lower() for c in df.columns])
df.createOrReplaceTempView("trips")

print(f"OBT_TRIPS: {df.count():,} filas | {len(df.columns)} columnas")

## a) Top 10 zonas de pickup por volumen mensual

In [ ]:
spark.sql("""
    SELECT pu_zone, source_year, source_month, COUNT(*) as trips
    FROM trips
    WHERE pu_zone IS NOT NULL
    GROUP BY pu_zone, source_year, source_month
    ORDER BY trips DESC
    LIMIT 10
""").show(truncate=False)

## b) Top 10 zonas de dropoff por volumen mensual

In [ ]:
spark.sql("""
    SELECT do_zone, source_year, source_month, COUNT(*) as trips
    FROM trips
    WHERE do_zone IS NOT NULL
    GROUP BY do_zone, source_year, source_month
    ORDER BY trips DESC
    LIMIT 10
""").show(truncate=False)

## c) Evolucion mensual de total_amount y tip_pct por borough

In [ ]:
spark.sql("""
    SELECT pu_borough, source_year, source_month,
           ROUND(AVG(total_amount), 2) as avg_total,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM trips
    WHERE pu_borough IS NOT NULL
    GROUP BY pu_borough, source_year, source_month
    ORDER BY pu_borough, source_year, source_month
""").show(50, truncate=False)

## d) Ticket promedio por service_type y mes

In [ ]:
spark.sql("""
    SELECT service_type, source_year, source_month,
           ROUND(AVG(total_amount), 2) as avg_total_amount
    FROM trips
    GROUP BY service_type, source_year, source_month
    ORDER BY service_type, source_year, source_month
""").show(50, truncate=False)

## e) Viajes por hora del dia y dia de semana (picos)

In [ ]:
spark.sql("""
    SELECT day_of_week, pickup_hour, COUNT(*) as trips
    FROM trips
    GROUP BY day_of_week, pickup_hour
    ORDER BY day_of_week, pickup_hour
""").show(200, truncate=False)

## f) p50/p90 de trip_duration_min por borough de pickup

In [ ]:
spark.sql("""
    SELECT pu_borough,
           ROUND(PERCENTILE_APPROX(trip_duration_min, 0.5), 2) as p50_duration,
           ROUND(PERCENTILE_APPROX(trip_duration_min, 0.9), 2) as p90_duration
    FROM trips
    WHERE pu_borough IS NOT NULL AND trip_duration_min > 0
    GROUP BY pu_borough
    ORDER BY p50_duration DESC
""").show(truncate=False)

## g) avg_speed_mph por franja horaria (6-9, 17-20) y borough

In [ ]:
spark.sql("""
    SELECT pu_borough,
           CASE
               WHEN pickup_hour BETWEEN 6 AND 9 THEN 'Morning Rush (6-9)'
               WHEN pickup_hour BETWEEN 17 AND 20 THEN 'Evening Rush (17-20)'
               ELSE 'Other'
           END as time_slot,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed
    FROM trips
    WHERE pu_borough IS NOT NULL AND avg_speed_mph IS NOT NULL AND avg_speed_mph < 100
    GROUP BY pu_borough, time_slot
    ORDER BY pu_borough, time_slot
""").show(50, truncate=False)

## h) Participacion por payment_type_desc y su relacion con tip_pct

In [ ]:
spark.sql("""
    SELECT payment_type_desc,
           COUNT(*) as trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_trips,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM trips
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc
    ORDER BY trips DESC
""").show(truncate=False)

## i) Rate codes con mayor trip_distance y total_amount

In [ ]:
spark.sql("""
    SELECT rate_code_desc,
           COUNT(*) as trips,
           ROUND(AVG(trip_distance), 2) as avg_distance,
           ROUND(AVG(total_amount), 2) as avg_total,
           ROUND(SUM(total_amount), 2) as sum_total
    FROM trips
    WHERE rate_code_desc IS NOT NULL
    GROUP BY rate_code_desc
    ORDER BY sum_total DESC
""").show(truncate=False)

## j) Mix yellow vs green por mes y borough

In [ ]:
spark.sql("""
    SELECT pu_borough, source_year, source_month, service_type,
           COUNT(*) as trips
    FROM trips
    WHERE pu_borough IS NOT NULL
    GROUP BY pu_borough, source_year, source_month, service_type
    ORDER BY pu_borough, source_year, source_month, service_type
""").show(50, truncate=False)

## k) Top 20 flujos PU→DO por volumen y ticket promedio

In [ ]:
spark.sql("""
    SELECT pu_zone, do_zone,
           COUNT(*) as trips,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM trips
    WHERE pu_zone IS NOT NULL AND do_zone IS NOT NULL
    GROUP BY pu_zone, do_zone
    ORDER BY trips DESC
    LIMIT 20
""").show(truncate=False)

## l) Distribucion de passenger_count y efecto en total_amount

In [ ]:
spark.sql("""
    SELECT CAST(passenger_count AS INT) as passengers,
           COUNT(*) as trips,
           ROUND(AVG(total_amount), 2) as avg_total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM trips
    WHERE passenger_count IS NOT NULL AND passenger_count > 0
    GROUP BY CAST(passenger_count AS INT)
    ORDER BY passengers
""").show(truncate=False)

## m) Impacto de tolls_amount y congestion_surcharge por zona

In [ ]:
spark.sql("""
    SELECT pu_zone,
           ROUND(AVG(tolls_amount), 2) as avg_tolls,
           ROUND(AVG(congestion_surcharge), 2) as avg_congestion,
           COUNT(*) as trips
    FROM trips
    WHERE pu_zone IS NOT NULL
    GROUP BY pu_zone
    HAVING COUNT(*) > 1000
    ORDER BY avg_congestion DESC
    LIMIT 20
""").show(truncate=False)

## n) Proporcion viajes cortos vs largos por borough y estacionalidad

In [ ]:
spark.sql("""
    SELECT pu_borough,
           CASE
               WHEN month IN (12, 1, 2) THEN 'Winter'
               WHEN month IN (3, 4, 5) THEN 'Spring'
               WHEN month IN (6, 7, 8) THEN 'Summer'
               ELSE 'Fall'
           END as season,
           ROUND(SUM(CASE WHEN trip_distance <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_short,
           ROUND(SUM(CASE WHEN trip_distance > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_long,
           COUNT(*) as total_trips
    FROM trips
    WHERE pu_borough IS NOT NULL AND trip_distance IS NOT NULL
    GROUP BY pu_borough, season
    ORDER BY pu_borough, season
""").show(50, truncate=False)

## o) Diferencias por vendor en avg_speed_mph y trip_duration_min

In [ ]:
spark.sql("""
    SELECT vendor_name,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed,
           ROUND(AVG(trip_duration_min), 2) as avg_duration,
           COUNT(*) as trips
    FROM trips
    WHERE vendor_name IS NOT NULL AND avg_speed_mph IS NOT NULL AND avg_speed_mph < 100
    GROUP BY vendor_name
""").show(truncate=False)

## p) Relacion metodo de pago y tip_amount por hora

In [ ]:
spark.sql("""
    SELECT payment_type_desc, pickup_hour,
           ROUND(AVG(tip_amount), 2) as avg_tip,
           COUNT(*) as trips
    FROM trips
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc, pickup_hour
    ORDER BY payment_type_desc, pickup_hour
""").show(100, truncate=False)

## q) Zonas con percentil 99 de duracion/distancia fuera de rango

In [ ]:
spark.sql("""
    SELECT pu_zone,
           ROUND(PERCENTILE_APPROX(trip_duration_min, 0.99), 2) as p99_duration,
           ROUND(PERCENTILE_APPROX(trip_distance, 0.99), 2) as p99_distance,
           COUNT(*) as trips
    FROM trips
    WHERE pu_zone IS NOT NULL AND trip_duration_min > 0
    GROUP BY pu_zone
    HAVING COUNT(*) > 1000
    ORDER BY p99_duration DESC
    LIMIT 20
""").show(truncate=False)

## r) Yield por milla (total_amount/trip_distance) por borough y hora

In [ ]:
spark.sql("""
    SELECT pu_borough, pickup_hour,
           ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) as yield_per_mile,
           COUNT(*) as trips
    FROM trips
    WHERE pu_borough IS NOT NULL AND trip_distance > 0.1
    GROUP BY pu_borough, pickup_hour
    ORDER BY pu_borough, pickup_hour
""").show(200, truncate=False)

## s) Cambios YoY en volumen y ticket promedio por service_type

In [ ]:
spark.sql("""
    WITH yearly AS (
        SELECT service_type, source_year,
               COUNT(*) as trips,
               ROUND(AVG(total_amount), 2) as avg_total
        FROM trips
        GROUP BY service_type, source_year
    )
    SELECT
        a.service_type, a.source_year,
        a.trips, a.avg_total,
        ROUND((a.trips - b.trips) * 100.0 / NULLIF(b.trips, 0), 2) as volume_change_pct,
        ROUND(a.avg_total - b.avg_total, 2) as ticket_change
    FROM yearly a
    LEFT JOIN yearly b ON a.service_type = b.service_type AND a.source_year = b.source_year + 1
    ORDER BY a.service_type, a.source_year
""").show(50, truncate=False)

## t) Dias con alta congestion_surcharge: efecto en total_amount vs dias normales

In [ ]:
spark.sql("""
    WITH daily AS (
        SELECT pickup_date,
               AVG(congestion_surcharge) as avg_congestion,
               AVG(total_amount) as avg_total,
               COUNT(*) as trips
        FROM trips
        WHERE congestion_surcharge IS NOT NULL AND pickup_date IS NOT NULL
        GROUP BY pickup_date
    )
    SELECT
        CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END as congestion_level,
        COUNT(*) as num_days,
        ROUND(AVG(avg_total), 2) as avg_total_amount,
        ROUND(AVG(trips), 0) as avg_daily_trips
    FROM daily
    GROUP BY CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END
""").show(truncate=False)